# The pinpoint-vs-area confound, and the fix

FPA-FOD stores each fire as a **point** (`LATITUDE`/`LONGITUDE`) but `FIRE_SIZE` describes an
**area**. At EPA Level III grain that mismatch mostly stays inside the polygon, so every published
number in [`06_analysis.ipynb`](06_analysis.ipynb) and [`07_natural_location.ipynb`](07_natural_location.ipynb)
is unaffected. Below ecoregion grain it stops being harmless.

**The problem.** A hexgrid fine enough to site mitigation is finer than most large fires. Crediting a
fire's whole acreage to the single hex containing its ignition point would make a hex-level
burned-area target measure *point-attribution error* rather than fire behavior — it would be
measuring the defect instead of the phenomenon.

**The fix is a join, not a new dataset.** FPA-FOD carries `MTBS_ID`, a foreign key into the MTBS
burned-area perimeters. It resolves for only 0.6% of fires nationally — but those fires hold **81.6%
of all burned acres**. The complement is not a problem: point-only fires average 14 acres, far
smaller than one hex, so crediting them wholly to their containing hex is accurate rather than a
concession.

This notebook demonstrates [`src/hex_burn.py`](../src/hex_burn.py), which implements that hybrid rule
and the acre-conservation invariant it rests on:

```
perimeter-backed fire -> split acres across hexes by intersected area
point-only fire       -> all acres to the containing hex
```

**Scope.** The three W5 proof-of-concept ecoregions — Klamath Mountains/California High North Coast
Range, Idaho Batholith, Blue Mountains — at H3 resolution 5.

In [1]:
import sys
import warnings

import geopandas as gpd
import numpy as np
import pandas as pd

sys.path.insert(0, "../src")
import hex_burn as hb
from config import ProjectConfig

warnings.filterwarnings("ignore")
cfg = ProjectConfig()
DATA = cfg.data

# The three contrasting CONUS ecoregions the W5 probe runs on. Chosen over the single
# largest region (AK Interior Forested Lowlands) because all three together are ~33%
# SMALLER than it, while adding cross-regime evidence — and because high-latitude MODIS
# carries cloud/snow/solar-zenith problems that would degrade the fuel signal itself.
POC_REGIONS = [
    "Klamath Mountains/California High North Coast Range",
    "Idaho Batholith",
    "Blue Mountains",
]

print(f"H3 res 5 hex: {hb.hex_area_acres(5):,.0f} acres ({hb.hex_area_acres(5) / 247.105:,.0f} km2)")
print(f"H3 res 4 hex: {hb.hex_area_acres(4):,.0f} acres")
print(f"H3 res 6 hex: {hb.hex_area_acres(6):,.0f} acres")

H3 res 5 hex: 62,494 acres (253 km2)
H3 res 4 hex: 437,462 acres
H3 res 6 hex: 8,928 acres


## Build the hexgrid

Hexes are **clipped to the region boundary** and carry `land_area_acres` — the area *inside* the
region, not the full hex. Using full hex area as a denominator would understate `burned_frac`
exactly at the edges. A hex straddling two regions is assigned to whichever holds more of it, so
`hex_id` stays a unique key.

In [2]:
eco = gpd.read_file(DATA / "us_eco_l3_state_boundaries")
eco = eco[eco["US_L3NAME"].isin(POC_REGIONS)].rename(columns={"US_L3NAME": "region"})
eco = eco[["region", "geometry"]]

# The shapefile ships polygons split by state, so one region is several rows.
print(f"source polygons: {len(eco)} for {eco['region'].nunique()} regions")

grid = hb.build_hexgrid(eco, resolution=hb.HEX_RES_MVP)

print(f"\nhexes built: {len(grid):,}  (unique hex_id: {grid['hex_id'].nunique():,})")
print(f"grid CRS: {grid.crs}  <- equal-area, required for honest area arithmetic\n")
print(
    grid.groupby("region")
    .agg(n_hexes=("hex_id", "size"), land_acres=("land_area_acres", "sum"))
    .round(0)
    .to_string()
)

source polygons: 8 for 3 regions

hexes built: 739  (unique hex_id: 739)
grid CRS: EPSG:5070  <- equal-area, required for honest area arithmetic

                                                     n_hexes  land_acres
region                                                                  
Blue Mountains                                           297  17039959.0
Idaho Batholith                                          244  14279358.0
Klamath Mountains/California High North Coast Range      198  11378718.0


## Load fires and the perimeters they point to

`MTBS_ID` is already carried in `fires_clean.parquet`, so no re-join against the SQLite source is
needed. Only the perimeters actually referenced by these fires are read — the full shapefile is
617 MB and reading all 30,930 geometries to use ~850 of them would be wasteful.

In [3]:
fires = pd.read_parquet(DATA / "fires_clean.parquet")
fires = fires[fires["region"].isin(POC_REGIONS)].copy()

wanted = set(fires["MTBS_ID"].fillna("").astype(str).str.strip()) - {""}
perims = gpd.read_file(DATA / "mtbs_perimeters" / "mtbs_perims_DD.shp")
perims["event_id"] = perims["event_id"].astype(str).str.strip()
perims = perims[perims["event_id"].isin(wanted)]

n_with_id = (fires["MTBS_ID"].fillna("").astype(str).str.strip() != "").sum()
print(f"fires in the three regions : {len(fires):,}")
print(f"total acres                : {fires['FIRE_SIZE'].sum():,.0f}")
print(f"carrying an MTBS_ID        : {n_with_id:,} ({100 * n_with_id / len(fires):.1f}% of fires)")
print(f"perimeters that resolve    : {len(perims):,}")

fires in the three regions : 69,572
total acres                : 15,808,832
carrying an MTBS_ID        : 1,031 (1.5% of fires)
perimeters that resolve    : 852


## Distribute acres — the hybrid rule

Two design decisions inside `build_hex_acres` are judgment calls rather than mechanics, and both
matter:

**Unresolved `MTBS_ID`s fall back to point attribution rather than being dropped.** Nationally 1,144
of 13,870 IDs do not resolve — 478 are agency-prefixed (`FS-`, `NP-`, `BL-`), a different identifier
scheme that matches nothing by construction, and 666 are geo-format IDs absent from the published
set. Discarding them would silently lose ~6.7M acres nationally, which is precisely the large-fire
mass this probe is about.

**Magnitude comes from `FIRE_SIZE`; only *shape* comes from the perimeter.** MTBS `burnbndac` agrees
with computed polygon area to ~0.1%, but the two sources are not identical and `FIRE_SIZE` is the
quantity every published project number is denominated in. Rescaling keeps this a **redistribution**
of existing acres, never a restatement of them.

In [4]:
hex_acres = hb.build_hex_acres(fires, perims, grid, resolution=hb.HEX_RES_MVP)

print(f"output rows: {len(hex_acres):,}  (one per fire x hex it touched)\n")
print(
    hex_acres.groupby("source")
    .agg(rows=("hex_acres", "size"), acres=("hex_acres", "sum"))
    .round(0)
    .to_string()
)

output rows: 69,836  (one per fire x hex it touched)

            rows       acres
source                      
perimeter   2935  14601170.0
point      66901   1086569.0


## The invariant: acres are conserved

For every fire the per-hex weights sum to 1, so distributed acres sum back to that fire's
`FIRE_SIZE`. This is load-bearing, not cosmetic — it is what will make the hex panel reconcile
**exactly** to `region_season_cause.parquet`, so any reconciliation failure downstream is a bug in
`hex_burn.py` rather than a modeling choice.

`assert_acres_conserved` raises and names the worst offender, because a silent violation here would
corrupt every downstream number while still looking entirely plausible.

Acres landing **outside** the grid are expected and are reported rather than absorbed: fires near a
region edge have footprints that leave the study extent.

In [5]:
rep = hb.coverage_report(fires, hex_acres)

print("Acre accounting")
print(f"  acres in            : {rep['acres_in']:>14,.0f}")
print(f"  landed on the grid  : {rep['acres_on_grid']:>14,.0f}   ({rep['pct_on_grid']:.2f}%)")
print(f"  fell outside extent : {rep['acres_outside']:>14,.0f}   (boundary-straddling fires)")
print()
print(f"  from perimeters     : {rep['acres_perimeter']:>14,.0f}   ({rep['pct_perimeter']:.2f}% of landed)")
print(f"  from points         : {rep['acres_point']:>14,.0f}")
print(f"  hexes touched       : {rep['n_hexes_touched']:>14,}")

hb.assert_acres_conserved(fires, hex_acres)
print("\nACRE CONSERVATION: PASS  (every in-grid fire sums to its FIRE_SIZE within 1e-6)")

Acre accounting
  acres in            :     15,808,832
  landed on the grid  :     15,687,738   (99.23%)
  fell outside extent :        121,093   (boundary-straddling fires)

  from perimeters     :     14,601,170   (93.07% of landed)
  from points         :      1,086,569
  hexes touched       :            739

ACRE CONSERVATION: PASS  (every in-grid fire sums to its FIRE_SIZE within 1e-6)


## How wrong would point attribution have been?

This is the confound quantified rather than asserted. For each perimeter-backed fire, how many hexes
does its footprint actually cover — and what share of its acres fall somewhere other than the hex
holding the ignition point?

In [6]:
per = hex_acres[hex_acres["source"] == "perimeter"]
spread = per.groupby("fire_key").agg(n_hex=("hex_id", "size"), acres=("hex_acres", "sum"))

multi = spread["n_hex"] > 1
print("Hexes spanned by a perimeter-backed fire:")
print(spread["n_hex"].describe().round(1).to_string())

print(f"\nspanning more than one hex : {multi.sum():,} of {len(spread):,} ({100 * multi.mean():.0f}%)")
print(
    f"acres in multi-hex fires   : {spread.loc[multi, 'acres'].sum():,.0f} "
    f"({100 * spread.loc[multi, 'acres'].sum() / spread['acres'].sum():.1f}% of perimeter acres)"
)

# Share of each fire's acres NOT in its single largest hex — what point attribution would misplace.
largest = per.groupby("fire_key")["hex_acres"].max()
misplaced = 1 - (largest / spread["acres"])
weighted = float((spread["acres"] * misplaced).sum() / spread["acres"].sum())
print(f"\nacre-weighted share of burned area outside the single largest hex: {100 * weighted:.1f}%")

Hexes spanned by a perimeter-backed fire:
count    955.0
mean       3.1
std        3.6
min        1.0
25%        1.0
50%        2.0
75%        3.0
max       26.0

spanning more than one hex : 636 of 955 (67%)
acres in multi-hex fires   : 13,385,011 (91.7% of perimeter acres)

acre-weighted share of burned area outside the single largest hex: 49.9%


## Finding

**Point attribution would have misplaced roughly half the burned area at this grain.** Two thirds of
perimeter-backed fires span more than one res-5 hex, and those fires carry ~92% of the
perimeter-backed acres. The median large fire covers 2 hexes and the largest covers 26. Weighted by
acres, **~50% of burned area falls outside the single hex holding the largest share of the fire** —
and the ignition point is not guaranteed to sit in that hex, so this is a *lower bound* on what
point attribution would have gotten wrong.

This is the empirical form of the confound: at ecoregion grain the point-vs-area mismatch is
absorbed inside the polygon, but at the grain a planner would actually site mitigation on, a
point-attributed target would be dominated by attribution error. **The corrected target is a
prerequisite for the probe, not a refinement of it.**

Note what this does *not* claim. It says nothing yet about whether burned area is predictable at hex
grain — only that a naive target would have measured the wrong thing. The predictive question is the
MVP's, and it needs the fuel-condition imagery rung to answer.

**Next:** aggregate `hex_acres` onto the hex x season x season-year panel and reconcile its totals
against `region_season_cause.parquet`. Exact agreement there is the end-to-end proof of the
conservation invariant demonstrated above.

## A worked example — the visual the confound deserves

One real fire, drawn twice: what the record stores, and what actually burned. This is the basis for
the W5 practice-talk visual (**V1**). The example is chosen near the *median* of the large-fire
distribution rather than the largest available — the point is what is typical, not what is dramatic.

In [7]:
big = fires[fires["MTBS_ID"].isin(set(perims["event_id"])) & (fires["FIRE_SIZE"] >= 1000)]
target = big["FIRE_SIZE"].median()
pick = big.iloc[(big["FIRE_SIZE"] - target).abs().argsort().iloc[0]]

n_hex = int((hex_acres["fire_key"] == pick["FOD_ID"]).sum())
print(f"median large fire in the PoC regions: {target:,.0f} acres")
print(f"\nchosen example: {pick.get('FIRE_NAME', 'n/a')}  ({pick['FIRE_YEAR']}, {pick['region']})")
print(f"  FIRE_SIZE     : {pick['FIRE_SIZE']:,.0f} acres")
print(f"  recorded as   : a single point at ({pick['LATITUDE']:.4f}, {pick['LONGITUDE']:.4f})")
print(f"  actually spans: {n_hex} res-5 hexes")

median large fire in the PoC regions: 4,500 acres

chosen example: TAILHOLT CREEK  (2006, Idaho Batholith)
  FIRE_SIZE     : 4,500 acres
  recorded as   : a single point at (45.0592, -115.6889)
  actually spans: 2 res-5 hexes


---

# Scaling up: the national grid

Everything above ran on three ecoregions — 2.3% of CONUS. The same primitive scales to the whole
record, and doing so is cheap: the grid is pure geometry and the perimeter intersection is roughly
linear in perimeter count.

**Alaska forced one change.** `hex_burn` originally hardcoded EPSG:5070 (CONUS Albers), which is not
valid at Alaska's latitudes — and Alaska carries **20.4% of all burned acres**, far too much to drop
from a national build. The CRS is now a parameter, and `build_national_acres` runs each landmass in
its own equal-area projection (5070 for CONUS, 3338 for Alaska) before concatenating. H3 ids are
globally unique, so the two halves need no re-keying.

**One data defect had to be repaired.** The Alaska Level III layer ships a self-intersecting polygon
in SE Alaska; dissolving it raises a GEOS side-location conflict at (−135.338, 57.255). This is the
same invalid geometry [`src/terraclimate.py`](../src/terraclimate.py) encountered — it dodged the
problem by never dissolving, but a hexgrid genuinely needs one geometry per region to tessellate and
clip against, so `build_hexgrid` calls `make_valid()` on invalid rows instead.

In [8]:
# ~100s over the full record. Writes two artifacts, so downstream work reloads
# rather than rebuilds. Skip if they already exist.
NAT_ACRES = DATA / "hex_acres_res5.parquet"
NAT_GRID = DATA / "hex_grid_res5.parquet"

if NAT_ACRES.exists() and NAT_GRID.exists():
    nat_acres = pd.read_parquet(NAT_ACRES)
    nat_grid = pd.read_parquet(NAT_GRID)
    print("loaded cached national artifacts")
else:
    all_fires = pd.read_parquet(DATA / "fires_clean.parquet")
    all_want = set(all_fires["MTBS_ID"].fillna("").astype(str).str.strip()) - {""}
    all_perims = gpd.read_file(DATA / "mtbs_perimeters" / "mtbs_perims_DD.shp")
    all_perims["event_id"] = all_perims["event_id"].astype(str).str.strip()
    all_perims = all_perims[all_perims["event_id"].isin(all_want)]

    nat_acres, nat_grid, coverage = hb.build_national_acres(
        all_fires, all_perims,
        DATA / "us_eco_l3_state_boundaries", DATA / "ak_eco_l3",
        resolution=hb.HEX_RES_MVP,
    )
    nat_acres.to_parquet(NAT_ACRES, index=False)
    nat_grid.to_parquet(NAT_GRID, index=False)
    print("built and cached national artifacts")

print(
    nat_grid.groupby("landmass")
    .agg(hexes=("hex_id", "size"), regions=("region", "nunique"),
         land_Macres=("land_area_acres", lambda s: s.sum() / 1e6))
    .round(1).to_string()
)
print(f"\ntotal: {len(nat_grid):,} hexes across {nat_grid['region'].nunique()} ecoregions")

loaded cached national artifacts
          hexes  regions  land_Macres
landmass                             
AK         6351       20        337.9
CONUS     29883       85       1809.5

total: 36,234 hexes across 105 ecoregions


## Does it still conserve acres at national scale?

The invariant that mattered on three regions matters more on 105: every fire's distributed acres must
still sum to its `FIRE_SIZE`. Below that, the acre accounting — and the one number that needs
interpreting rather than just reporting.

In [9]:
all_fires = pd.read_parquet(DATA / "fires_clean.parquet")
hb.assert_acres_conserved(all_fires, nat_acres)
print("ACRE CONSERVATION (national): PASS\n")

landed = nat_acres["hex_acres"].sum()
total = all_fires["FIRE_SIZE"].sum()
print(f"  acres in record     : {total:>14,.0f}")
print(f"  landed on the grid  : {landed:>14,.0f}   ({100 * landed / total:.2f}%)")
print(f"  fell outside extent : {total - landed:>14,.0f}   ({100 * (total - landed) / total:.2f}%)")
print(f"  rows                : {len(nat_acres):>14,}")

by_src = nat_acres.groupby("source")["hex_acres"].sum()
print(f"\n  perimeter-derived   : {100 * by_src.get('perimeter', 0) / landed:.1f}% of landed acres")

ACRE CONSERVATION (national): PASS

  acres in record     :    179,405,808
  landed on the grid  :    178,711,432   (99.61%)
  fell outside extent :        694,377   (0.39%)
  rows                :      2,260,347

  perimeter-derived   : 78.1% of landed acres


## Where do the off-grid acres go?

0.38% of acres do not land on any hex. That is not a bug and it is not noise to be waved past — it is
a **boundary effect with a clear signature**, and it should be characterized rather than footnoted.

A fire's perimeter is clipped to the hexgrid, and the hexgrid is clipped to the ecoregion boundary.
When a fire burns across that boundary — or, for coastal regions, out over water — the part of its
footprint outside the region has nowhere to land.

In [10]:
fy = all_fires.set_index("FOD_ID")
by_region = (
    nat_acres.assign(region=nat_acres["fire_key"].map(fy["region"]))
    .groupby("region")["hex_acres"].sum()
)
recon = pd.DataFrame({
    "source_acres": all_fires.groupby("region")["FIRE_SIZE"].sum(),
    "hex_acres": by_region,
}).dropna()
recon["pct_landed"] = 100 * recon["hex_acres"] / recon["source_acres"]
recon["acres_lost"] = recon["source_acres"] - recon["hex_acres"]

print(f"regions at >=99% of acres : {(recon['pct_landed'] >= 99).sum()} of {len(recon)}")
print(f"national acres lost       : {recon['acres_lost'].sum():,.0f} "
      f"({100 * recon['acres_lost'].sum() / recon['source_acres'].sum():.2f}%)\n")
print("largest losses by region:")
print(recon.nlargest(8, "acres_lost")[["source_acres", "hex_acres", "pct_landed"]].round(1).to_string())

regions at >=99% of acres : 82 of 104
national acres lost       : 675,470 (0.38%)

largest losses by region:
                                         source_acres  hex_acres  pct_landed
region                                                                      
Western Gulf Coastal Plain                  1154754.9  1064200.7        92.2
Southern Coastal Plain                      4052479.1  3977684.9        98.2
Middle Atlantic Coastal Plain                777777.7   708989.1        91.2
Interior Highlands                          5769510.2  5709130.7        99.0
Madrean Archipelago                         2312160.4  2277864.2        98.5
Southern California/Northern Baja Coast     2715745.4  2688598.6        99.0
Southwestern Tablelands                     6253093.9  6229901.9        99.6
Cross Timbers                               2324624.9  2304316.6        99.1


### Finding — the loss is coastal, and it is small

Six of the eight largest losses are **coastal plain** ecoregions — Western Gulf, Southern, and Middle
Atlantic Coastal Plain, plus Coast Range. Their fires burn across a boundary that is partly
*shoreline*, so the missing acres are largely footprint over water. Interior Highlands and Madrean
Archipelago are the interior exceptions, both losing ~1% across land borders with neighbouring
ecoregions.

**Scale check:** 22 of 104 regions fall below 99% acre capture, but the national loss is
**0.38%**, and the regions carrying the most burned area reconcile to ~100%. For the ranking target
this is immaterial — a hex's own acres are correct regardless of what happened outside the region.
It would matter if the project ever claimed *regional totals* from the hex panel; for that, use the
published `region_season_cause.parquet` figures, which are not clipped.

**A related distinction, so the numbers are not misread.** `region_season_cause.parquet` totals
146.1M acres against `fires_clean.parquet`'s 179.4M. That gap is the *attributed-vs-total* split from
the existing pipeline — the published cell figures cover cause-attributed acres — and has nothing to
do with the hexgrid clipping described above. Two different subtractions; do not add them together.

## What the national grid unlocks

| | three PoC regions | national |
|---|---|---|
| hexes (res 5) | 739 | **36,234** |
| ecoregions | 3 | **105** |
| share of CONUS | 2.3% | 100% (+ Alaska) |
| acres on grid | 99.23% | **99.61%** |

Two uses, deliberately separate:

- **Descriptive and visual — use the national grid.** It needs no imagery, so it is unblocked today,
  and it gives the honest national denominator for any statement about coverage or sparsity.
- **The predictive MVP — stays at three regions.** The go/no-go is whether fuel condition beats
  climatology; 739 hexes over 21 years is ample to detect that. Scaling the model to 36,234 hexes
  would multiply the *blocked* imagery acquisition ~42× without changing the answer. There is also a
  modeling argument against it: pooling 105 ecoregions with different fire regimes into one model
  risks exactly the failure that sank the ecoregion-grain climate rung, where regionally
  heterogeneous relationships averaged out to nothing.

National rollout is what happens **after** the proof of concept says imagery helps.